[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap11_cdi_isentos.ipynb)

# Capítulo 11: Renda fixa II: bancários e crédito privado

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você baixa o CDI do Banco Central, transforma qualquer "% do CDI" em valor líquido de imposto e descobre quando um título isento (LCI, LCA, CRI, CRA) rende mais que um CDB.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. O CDI de hoje
Série 4389 do Banco Central: taxa DI anualizada, base 252 dias úteis.

In [ ]:
cdi_ano = sgs(4389, inicio=(dt.date.today() - dt.timedelta(days=40)).strftime('%d/%m/%Y')) / 100
CDI = float(cdi_ano.iloc[-1])
print('CDI em', cdi_ano.index[-1].strftime('%d/%m/%Y'), ':', pct(CDI))

## 2. Do "% do CDI" ao valor líquido
Tabela regressiva do IR na renda fixa (Lei 11.033/2004): 22,5% até 180 dias, 20% até 360, 17,5% até 720 e 15% acima.

In [ ]:
def aliquota_ir(dias):
    return 0.225 if dias <= 180 else 0.20 if dias <= 360 else 0.175 if dias <= 720 else 0.15

def liquido(valor, pct_cdi, dias_corridos, isento=False, cdi=CDI):
    dias_uteis = round(dias_corridos * 252 / 365)
    diaria = (1 + cdi) ** (1/252) - 1
    bruto = valor * ((1 + diaria * pct_cdi) ** dias_uteis - 1)
    ir = 0 if isento else bruto * aliquota_ir(dias_corridos)
    return bruto - ir

VALOR = 10_000   # MUDE AQUI
for dias in (180, 365, 730, 1095):
    cdb = liquido(VALOR, 1.00, dias)
    lci = liquido(VALOR, 0.90, dias, isento=True)
    print(f'{dias} dias: CDB 100% do CDI rende {brl(cdb)} líquido; LCI 90% do CDI rende {brl(lci)}')

## 3. A taxa equivalente
Para empatar com um isento, o tributado precisa pagar: % do isento ÷ (1 − alíquota).

In [ ]:
PCT_ISENTO = 0.90   # MUDE AQUI
for dias, nome in ((180, 'até 180 dias'), (360, 'de 181 a 360 dias'), (720, 'de 361 a 720 dias'), (721, 'acima de 720 dias')):
    print(f'{nome}: o CDB precisa pagar pelo menos {pct(PCT_ISENTO/(1-aliquota_ir(dias)),1)} do CDI')

## 4. O limite do FGC
O Fundo Garantidor de Créditos cobre até R\$ 250 mil por CPF e por conglomerado, com teto de R\$ 1 milhão a cada quatro anos, somando principal e juros. Confira o quanto você teria daqui a alguns anos.

In [ ]:
APLICADO = 220_000   # MUDE AQUI
for anos in (1, 2, 3):
    v = APLICADO + liquido(APLICADO, 1.0, 365*anos) / (1 - aliquota_ir(365*anos))  # saldo bruto
    aviso = 'acima do limite!' if v > 250_000 else 'dentro do limite'
    print(f'Em {anos} ano(s): {brl(v)} ({aviso})')

Compare ofertas reais na [calculadora LCI x CDB](https://www.traderbrasil.com/blog/calculadora-lci-x-cdb-qual-rende-mais.php) e leia o guia [Selic e CDI](https://www.traderbrasil.com/blog/selic-e-cdi-como-funcionam-e-como-interpretar-rentabilidade.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 11 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap11/

Achou um erro? Escreva para contato@traderbrasil.com.